# ProductionIncidents

**Module 12 · Lesson 12.9**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why AI Incidents Are Different
- Severity and the Clock
- Declare and Run the Incident
- Triage: What Changed?
- Mitigate First: The Rollback Levers
- The AI-Specific Runaways

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## 3:17 AM, all green, and very much on fire

> 💡 **Info**
>
> Your phone buzzes at **3:17 AM**. The alerts are ugly: the assistant is recommending a competitor, the cost dashboard shows the whole day’s budget gone in two hours, and… every request is returning a clean **HTTP 200**. You open the ops dashboard and it is **green, top to bottom**. Uptime fine, error rate zero, latency normal. And yet the thing is very much on fire. That is what makes an AI incident its own craft: a normal outage 500s and *stops*, but an AI app keeps **answering** — confidently, plausibly, and wrong — while every signal a normal dashboard watches stays green. Handling it well is a lifecycle with a shape: **detect** the gray failure, **classify** its severity, put someone **in charge**, **triage** what changed, **mitigate** to stop the bleeding *before* you know the root cause, and then **learn** from it without blaming anyone. This lesson builds that whole loop — and you can run every piece with no cloud, no provider, and no API key.

### 🎯 What you will be able to do after this lesson

- **Build** an AI incident response — a gray-failure detector, a severity classifier with the MTTD/MTTR clock, a role assigner, a what-changed triage, the rollback levers, a retry-storm breaker, and a postmortem — as runnable models plus an illustrative on-call runbook.

- **Compare** a traditional (binary) failure with an AI gray failure, and mitigation-first vs root-cause-first response.

- **Operate** the incident-command roles (IC / Ops / Comms), the five rollback levers, and a blameless postmortem with a version snapshot.

- **Evaluate** an incident response: was it detected fast, commanded cleanly, mitigated before diagnosis, and learned from blamelessly?

> 📦 **Info**
>
> ✅ Before you startIn **12.8** you built the golden signals — quality, cost, TTFT — that *detect* a gray failure; this lesson is what you do the moment one breaches. In **12.2** you built retries and a circuit breaker; here the breaker is the mitigation that stops a retry storm from becoming a cost incident. In **12.6** canary deploys shifted traffic between revisions; the fastest incident mitigation is that same lever. Cost optimization is **Module 13**; the eval set a postmortem action item adds is **Lesson 14.4**; production readiness closes the module in **Lesson 12.10**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🚑 **Analogy**
>
> Running an AI incident is being the **emergency room, not the passer-by**. A passer-by sees someone upright and walking and assumes they are fine (the green ops dashboard: it is up, it is fast). The ER runs the actual vitals, tags the patient by how urgent they are (severity), puts one doctor in charge of the room (the incident commander), asks “what changed?” before cutting (triage), and *stabilizes the patient before diagnosing the disease* (mitigate before root cause). **Where it breaks down:** an ER treats one patient; your incident can be hitting every user at once, which is exactly why you mitigate broadly first and investigate the single cause after.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Green dashboards mean no incident.”** A gray failure returns a clean 200 with a wrong answer; uptime monitoring is blind to a quality, cost, or safety regression. You detect it with the golden signals.
> - **“Find the root cause first.”** No — mitigate to restore users first (roll back, fail over, kill switch), diagnose second. Time-to-mitigate is what users feel; the root cause can wait for the calm.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that turn a bad hour into a disaster. **The Incident Commander going head-down to type the fix** — now nobody is holding the big picture, tracking the timeline, or deciding; the IC coordinates, the Ops lead executes. And **blind retries against a failing model** — each retry piles more load on an already-degraded provider and burns the budget, a retry storm that feeds on itself. Use a circuit breaker and a budget cap, not more retries.

---

## 🎯 Concept 1: Why AI Incidents Are Different

### Why AI Incidents Are Different

A traditional failure is binary: it 500s and you roll back. An AI incident often returns a clean HTTP 200 with a confidently wrong answer - a gray failure the uptime monitor cannot see. You detect it with quality, cost, and safety signals, not uptime.

Start with why the normal incident playbook goes blind. A traditional software failure is **binary**: the endpoint returns a 500, the alert fires, you roll back, done. An AI incident is usually not like that. The endpoint returns **HTTP 200**, the JSON is valid, the response reads plausibly — and it is confidently *wrong*: off-brand, hallucinated, leaking PII, or quietly burning ten times the usual cost. These are **gray failures**: by every signal a normal dashboard watches (is it up, fast, erroring?) the service is perfectly healthy, while it fails the user on the one axis that matters. That is why AI incidents have a fourth shape ops has no gauge for — **quality, cost, safety, and latency** can each degrade while uptime stays green — and why detection has to come from the **quality, cost, and safety signals** you built in 12.8, not from uptime. The block contrasts an uptime monitor and an AI health check on the same batch, keyless.

> 🛡️ **Analogy**
>
> It is a **smoke detector that cannot smell gas**. Your smoke alarm is doing its job perfectly — no smoke, no beep, all clear — while a colorless, odorless gas leak fills the house. The alarm is not broken; it was simply never built to detect *this* failure. The uptime monitor is that smoke alarm: it faithfully reports 200s and low latency while the assistant quietly poisons every answer. You need a different sensor (a quality/cost/safety check) for the failure the first one cannot see.

Every request is returning HTTP 200, error rate is zero, latency is normal — but the assistant is giving wrong prices. What does the uptime monitor report?

**📝 `01_gray_failures.py`** - *runnable*

In [ ]:
# A traditional bug 500s and stops. An AI incident returns HTTP 200 with valid JSON
# that is confidently WRONG - a "gray failure" the uptime monitor cannot see.
# (each response is HTTP 200; the AI health check inspects quality / PII / cost)
responses = [
    {"id": "r1", "status": 200, "quality": 8.6, "has_pii": False, "cost_usd": 0.011},
    {"id": "r2", "status": 200, "quality": 3.1, "has_pii": False, "cost_usd": 0.012},   # hallucination
    {"id": "r3", "status": 200, "quality": 8.2, "has_pii": True,  "cost_usd": 0.010},   # PII leak
    {"id": "r4", "status": 200, "quality": 8.4, "has_pii": False, "cost_usd": 0.013},
    {"id": "r5", "status": 200, "quality": 8.0, "has_pii": False, "cost_usd": 0.190},   # cost spike
    {"id": "r6", "status": 200, "quality": 2.8, "has_pii": False, "cost_usd": 0.012},   # hallucination
]
def ai_health(r):
    if r["has_pii"]:        return "PII LEAK"
    if r["quality"] < 5.0:  return "HALLUCINATION"
    if r["cost_usd"] > 0.10: return "COST SPIKE"
    return None

up = sum(1 for r in responses if r["status"] == 200)
print("UPTIME MONITOR (is it up / fast / erroring?):")
print("  {}/{} responses returned HTTP 200 -> 0 incidents, dashboard GREEN".format(up, len(responses)))
print()
print("AI HEALTH CHECK (is the answer any good?):")
incidents = [(r["id"], ai_health(r)) for r in responses if ai_health(r)]
for rid, kind in incidents:
    print("  {}: HTTP 200, but {}".format(rid, kind))
print("  -> {} gray failures the uptime monitor never saw".format(len(incidents)))
print()
print("An AI incident does not 500 and stop; it keeps answering, 200 and wrong.")
print("You detect it with quality / cost / safety signals (Lesson 12.8), not uptime.")

# Output:
# UPTIME MONITOR (is it up / fast / erroring?):
#   6/6 responses returned HTTP 200 -> 0 incidents, dashboard GREEN
#
# AI HEALTH CHECK (is the answer any good?):
#   r2: HTTP 200, but HALLUCINATION
#   r3: HTTP 200, but PII LEAK
#   r5: HTTP 200, but COST SPIKE
#   r6: HTTP 200, but HALLUCINATION
#   -> 4 gray failures the uptime monitor never saw
#
# An AI incident does not 500 and stop; it keeps answering, 200 and wrong.
# You detect it with quality / cost / safety signals (Lesson 12.8), not uptime.

- The **uptime monitor** sees all six responses return HTTP 200 — zero incidents, dashboard green.
- The **AI health check** inspects the answers and flags **four gray failures**: two hallucinations, one PII leak, one cost spike.
- Every one of those is a real incident that the uptime monitor *never saw*, because the requests all succeeded at the transport layer.
- The lesson: an AI incident keeps answering, 200 and wrong — detect it with quality, cost, and safety signals (12.8), not uptime.

#### 💡 What just happened

⚡ What just happened? A traditional failure is binary (it 500s, you roll back); an AI incident often returns a clean 200 with a confidently wrong answer - a gray failure uptime monitoring is blind to. Tradeoff / the framing for the lesson: you detect AI incidents with the golden signals (quality, cost, safety), and everything that follows - classify, command, triage, mitigate, learn - is how you run the response once one fires.

- An ops monitor shows all six responses green (HTTP 200); toggle the AI health check on
- The same batch lights up 4 gray failures - 2 hallucination, 1 PII, 1 cost - the uptime monitor never saw

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Severity and the Clock

### Severity and the Clock

Classify by impact so you respond proportionally: PII/safety/outage/cost = SEV1, quality drop = SEV2. Then measure the two clocks - MTTD (how long it was broken before you knew) and MTTR (detect to recover). A short MTTD predicts a short MTTR.

Not every issue is a 3 AM wake-up call, so **classify first** and respond proportionally. A common scheme runs SEV1 to SEV4: **SEV1** is critical — a total outage, a safety or legal risk, a PII leak, a cost runaway — and pages on-call immediately. **SEV2** is major — a big error-rate jump, a two-point quality drop, a hallucination spike — the “silent killer” traditional monitoring misses. **SEV3** is minor with a workaround, **SEV4** is cosmetic. Each level carries a response-time target so nobody debates urgency at 3 AM. Then there are the two numbers that measure how you did: **MTTD** (mean time to detect — how long the system was broken before anyone knew) and **MTTR** (mean time to recover — detection to full resolution). For an AI app the dangerous one is usually **MTTD**: a gray failure runs silently for hours before detection, so the undetected window dwarfs the fix. Improving detection is the highest-leverage move, because a short MTTD predicts a short MTTR. The block classifies signals and reads both clocks, keyless.

> 🏥 **Analogy**
>
> It is a **hospital triage tag and a stopwatch**. When patients flood in, the nurse does not treat whoever shouts loudest — each gets a colored tag by how urgent they truly are (severity), so the critical case jumps the queue and the sprained wrist waits. And two clocks decide the outcome: how long the patient was deteriorating before anyone triaged them (MTTD), and how long from triage to stable (MTTR). A patient who bleeds unnoticed in the waiting room for hours is in far more danger than the length of the surgery suggests — the undetected time is the killer.

A quality drop ran silently for four hours, then was fixed in about thirty minutes. Where was the incident’s time really lost?

**📝 `02_severity_clock.py`** - *runnable*

In [ ]:
# Classify by impact so you respond proportionally - then measure the two clocks.
def classify(sig):
    # SEV1: critical (safety/legal/outage/cost runaway)
    if sig.get("pii_leak") or sig.get("safety_violation") or sig.get("total_outage"): return "SEV1"
    if sig.get("cost_per_hour", 0) > 100: return "SEV1"
    # SEV2: major (the "silent killer" - quality/hallucination/big errors)
    if sig.get("error_rate_pct", 0) > 20: return "SEV2"
    if sig.get("quality_drop", 0) >= 2:   return "SEV2"
    if sig.get("hallucination_pct", 0) > 15: return "SEV2"
    if sig.get("cost_per_hour", 0) > 50:  return "SEV2"
    # SEV3: minor
    if sig.get("error_rate_pct", 0) > 5:  return "SEV3"
    if sig.get("quality_drop", 0) >= 1:   return "SEV3"
    return "SEV4"

TARGETS = {"SEV1": "ack 5m / mitigate 30m", "SEV2": "ack 15m / mitigate 2h",
           "SEV3": "ack 1h / mitigate 1d", "SEV4": "ack 1d / mitigate 1wk"}
cases = [
    ("PII in a chatbot reply",        {"pii_leak": True}),
    ("Cost at $80/hour",              {"cost_per_hour": 80}),
    ("Quality dropped 2.5 points",    {"quality_drop": 2.5}),
    ("Error rate 8%",                 {"error_rate_pct": 8}),
    ("Slightly slower responses",     {"latency_p99_ms": 5000}),
]
print("Severity from signals:")
for name, sig in cases:
    sev = classify(sig)
    print("  {:<28} -> {}  ({})".format(name, sev, TARGETS[sev]))
print()
# The two clocks. MTTD = how long it was broken before anyone knew. MTTR = detect -> resolve.
# Timeline in minutes from when the incident BEGAN (a silent SEV2 quality drop):
began, detected, resolved = 0, 240, 275
mttd = detected - began
mttr = resolved - detected
print("The clock on a silent SEV2 (quality quietly dropped):")
print("  began t={}m   detected t={}m   resolved t={}m".format(began, detected, resolved))
print("  MTTD (broken before anyone knew) = {}m".format(mttd))
print("  MTTR (detect -> recover)         = {}m".format(mttr))
print("  The kill was the {}m MTTD, not the {}m fix. A short MTTD predicts a short MTTR:".format(mttd, mttr))
print("  detection is the highest-leverage number to improve.")

# Output:
# Severity from signals:
#   PII in a chatbot reply       -> SEV1  (ack 5m / mitigate 30m)
#   Cost at $80/hour             -> SEV2  (ack 15m / mitigate 2h)
#   Quality dropped 2.5 points   -> SEV2  (ack 15m / mitigate 2h)
#   Error rate 8%                -> SEV3  (ack 1h / mitigate 1d)
#   Slightly slower responses    -> SEV4  (ack 1d / mitigate 1wk)
#
# The clock on a silent SEV2 (quality quietly dropped):
#   began t=0m   detected t=240m   resolved t=275m
#   MTTD (broken before anyone knew) = 240m
#   MTTR (detect -> recover)         = 35m
#   The kill was the 240m MTTD, not the 35m fix. A short MTTD predicts a short MTTR:
#   detection is the highest-leverage number to improve.

- The **classifier** maps signals to severity: a PII leak is **SEV1**; an $80/hour cost or a two-point quality drop is **SEV2**; an 8-percent error rate is SEV3; a slight slowdown is SEV4.
- Each severity carries a **response-time target** (ack and mitigate windows) so urgency is never debated live.
- The **two clocks**: on a silent quality drop, **MTTD** was the long window it ran undetected and **MTTR** was the quick fix after detection.
- The kill was the MTTD, not the fix — so **detection is the highest-leverage number**, and a short MTTD predicts a short MTTR.

#### 💡 What just happened

⚡ What just happened? Classify by impact (SEV1 for PII/safety/outage/cost, SEV2 for a quality drop) so you respond proportionally, and track the two clocks: MTTD (broken before anyone knew) and MTTR (detect to recover). Tradeoff: for an AI app MTTD usually dominates, so investing in detection beats speeding up the fix. Both are for finding process bottlenecks, never for scoring an engineer. Next: who runs the response.

- Feed signals to a classifier that lights SEV1-4 with response targets
- A timeline bar shows a long MTTD then a short MTTR; the silent window dwarfs the fix

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Declare and Run the Incident

### Declare and Run the Incident

In a SEV1 you do not want ten people typing gcloud at once. Assign three separated roles: an Incident Commander who coordinates and decides (but does not fix), an Operations lead who executes, and a Communications lead who owns the status page.

When a SEV1 or SEV2 is declared, the failure mode is *chaos*: ten engineers in a channel, three of them typing conflicting `gcloud` commands, nobody sure who decided what. The fix is to **declare the incident and assign roles**. The Google SRE model uses three, deliberately separated: the **Incident Commander (IC)** coordinates the response, assigns roles, and makes the decisions — and does *not* do hands-on troubleshooting, because an IC head-down in a terminal stops holding the big picture. The **Operations (Tech) lead** drives the diagnosis and runs the actual mitigation commands. The **Communications lead** owns the status page and the stakeholder and customer updates, so the responders are not context-switching to write announcements. One **incident channel** is the single source of truth. This separation of duties is what keeps a high-pressure response coherent. The block assigns actions to roles and flags the anti-pattern, keyless.

> 🛩️ **Analogy**
>
> It is an **air-traffic controller, not a pilot**. The controller in the tower does not fly any of the planes — the moment they grab a yoke, they stop watching the other twenty aircraft and a near-miss becomes a collision. Their whole value is staying *out* of the cockpit: seeing the whole sky, sequencing the landings, making the calls. The Incident Commander is that controller. The pilots (the Ops lead) fly the planes; the tower (the IC) runs the airspace; and the airline’s press office (the Comms lead) talks to the passengers.

It is a SEV1. The Incident Commander’s instinct is to jump into the terminal and run the rollback themselves. Why is that a mistake?

**📝 `03_incident_roles.py`** - *runnable*

In [ ]:
# In a SEV1 you do not want ten people typing gcloud at once. Assign three SEPARATED roles.
ROLES = {
    "Incident Commander": "coordinates, assigns roles, decides - does NOT do hands-on fixes",
    "Operations Lead":    "drives the diagnosis and runs the mitigation commands",
    "Communications Lead":"owns the status page and stakeholder / customer updates",
}
def owner(action):
    a = action.lower()
    if "decide" in a or "coordinate" in a or "assign" in a: return "Incident Commander"
    if "run " in a or "read the traces" in a or "roll back" in a or "execute" in a: return "Operations Lead"
    if "status" in a or "notify" in a or "customer" in a: return "Communications Lead"
    return "Operations Lead"

print("Roles in a declared SEV1:")
for r, duty in ROLES.items():
    print("  {:<20} {}".format(r, duty))
print()
actions = [
    "Decide whether to roll back",
    "Run the traffic rollback command",
    "Read the traces to find the cause",
    "Post the status-page update",
    "Notify the enterprise customers",
]
print("Who owns each action:")
for a in actions:
    print("  {:<38} -> {}".format(a, owner(a)))
print()
# The anti-pattern: the IC going head-down to type the fix.
print("Anti-pattern check:")
print("  'The Incident Commander personally types the gcloud fix'")
print("  -> REJECTED: the IC coordinates and decides; the Ops Lead executes.")
print("  An IC head-down in a terminal loses the big picture the incident needs.")

# Output:
# Roles in a declared SEV1:
#   Incident Commander   coordinates, assigns roles, decides - does NOT do hands-on fixes
#   Operations Lead      drives the diagnosis and runs the mitigation commands
#   Communications Lead  owns the status page and stakeholder / customer updates
#
# Who owns each action:
#   Decide whether to roll back            -> Incident Commander
#   Run the traffic rollback command       -> Operations Lead
#   Read the traces to find the cause      -> Operations Lead
#   Post the status-page update            -> Communications Lead
#   Notify the enterprise customers        -> Communications Lead
#
# Anti-pattern check:
#   'The Incident Commander personally types the gcloud fix'
#   -> REJECTED: the IC coordinates and decides; the Ops Lead executes.
#   An IC head-down in a terminal loses the big picture the incident needs.

- Three **separated roles**: the **Incident Commander** coordinates and decides, the **Operations lead** executes, the **Communications lead** owns comms.
- Each action maps to one owner: *decide to roll back* is the IC; *run the rollback* and *read the traces* are the Ops lead; *post status* and *notify customers* are Comms.
- The **anti-pattern** is explicit: “the IC personally types the fix” is **rejected** — the IC coordinates, the Ops lead executes.
- One incident channel is the single source of truth, so the response stays coherent under pressure.

#### 💡 What just happened

⚡ What just happened? A declared SEV1 gets three separated roles: an Incident Commander who coordinates and decides (not fixes), an Operations lead who executes, and a Communications lead who owns the status page. Tradeoff: role separation costs a couple of people who are not typing fixes, in exchange for a response that does not descend into chaos. The IC staying out of the terminal is the whole point. Next: how the Ops lead finds what broke.

- Drag actions (decide, run the fix, post status) onto IC / Ops / Comms
- The IC-types-the-fix drop is rejected: the IC coordinates, the Ops lead executes

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Triage: What Changed?

### Triage: What Changed?

Bisect to the culprit layer - model, prompt, data, cache, or infra - by asking what changed. Score each recent change by whether it explains the symptoms and whether its timing matches the onset. The telemetry from 12.8 is how you bisect.

With someone in charge, the Ops lead has to **localize the fault** fast, and the fastest question is **“what changed?”** An AI app can break at five layers: the **model** (the provider bumped a version), the **prompt** (a deploy shipped), the **data** (a RAG reindex, stale embeddings), the **cache** (stale or wrong semantic matches), or the **infra** (5xx, latency, a scaling failure). You bisect by lining up the *symptoms* against the *recent changes*, scoring each change on two things: does it **explain the symptom** (a prompt change hits quality, not latency), and does its **timing match the onset** (the drop started when the change shipped)? The telemetry from 12.8 — traces, the golden signals, per-model and per-team slices — is what lets you rule layers in and out. A quality-only drop with flat errors, cost, and latency, coinciding with a deploy twenty minutes ago, points hard at the prompt, not a three-day-old model bump. The block runs that bisect, keyless.

> 🩺 **Analogy**
>
> It is a **doctor asking “what did you change?”** A patient turns up with a sudden rash. The good doctor does not run every test in the building; they ask what is new — a new soap this morning, a new medication three months ago, a new food last week. The soap that started exactly when the rash did is the suspect; the three-month-old medication is not, because the timing does not fit. Triage is that question: of everything that changed, which one both *could* cause this symptom and started *when* the symptom did?

Quality dropped twenty minutes ago; errors, cost, and latency are flat. A prompt deploy shipped twenty minutes ago; the model was bumped three days ago. Which is the culprit?

**📝 `04_triage_what_changed.py`** - *runnable*

In [ ]:
# Triage = bisect to the culprit LAYER by asking "what changed?". Score each recent
# change by whether it explains the symptoms AND whether its timing matches the onset.
onset_min_ago = 20   # quality started dropping ~20 minutes ago
changes = [
    {"layer": "PROMPT", "what": "prompt deploy",       "age_min": 20,   "affects": "quality"},
    {"layer": "MODEL",  "what": "provider version bump","age_min": 4320, "affects": "quality"},
    {"layer": "DATA",   "what": "RAG reindex",          "age_min": 1440, "affects": "quality"},
    {"layer": "INFRA",  "what": "autoscaler config",    "age_min": 30,   "affects": "latency"},
]
symptoms = {"quality": True, "error_rate": False, "cost": False, "latency": False}

def score(c):
    s = 0
    if symptoms.get(c["affects"]):                 s += 2   # explains the symptom
    if abs(c["age_min"] - onset_min_ago) <= 15:    s += 2   # timing matches the onset
    elif c["age_min"] <= 60:                       s += 1
    return s

print("Symptoms: quality DOWN; error rate / cost / latency all FLAT.")
print()
print("Rank recent changes by (explains symptom) + (timing matches onset):")
ranked = sorted(changes, key=score, reverse=True)
for c in ranked:
    print("  {:<6} {:<22} {:>5}m ago   score {}".format(c["layer"], c["what"], c["age_min"], score(c)))
print()
top = ranked[0]
print("Bisect: INFRA ruled out (latency flat), MODEL / DATA changes are too old to be the onset,")
print("so the culprit is the {} ({}, {}m ago) - it explains a quality-only drop AND matches the timing.".format(top["layer"], top["what"], top["age_min"]))
print("Confirm by rolling that one change back (next step) before hunting the exact line.")

# Output:
# Symptoms: quality DOWN; error rate / cost / latency all FLAT.
#
# Rank recent changes by (explains symptom) + (timing matches onset):
#   PROMPT prompt deploy             20m ago   score 4
#   MODEL  provider version bump   4320m ago   score 2
#   DATA   RAG reindex             1440m ago   score 2
#   INFRA  autoscaler config         30m ago   score 2
#
# Bisect: INFRA ruled out (latency flat), MODEL / DATA changes are too old to be the onset,
# so the culprit is the PROMPT (prompt deploy, 20m ago) - it explains a quality-only drop AND matches the timing.
# Confirm by rolling that one change back (next step) before hunting the exact line.

- The symptoms — **quality down, everything else flat** — are scored against each recent change on two axes.
- Each change earns points for **explaining the symptom** (affects quality) and for its **timing matching the onset** (about twenty minutes ago).
- The **prompt deploy** scores highest: it is a quality-affecting change that shipped right when the drop began.
- Infra is ruled out (latency flat), the model and data changes are too old — so confirm the prompt by **rolling it back** (next step) before hunting the exact line.

#### 💡 What just happened

⚡ What just happened? Triage bisects to the culprit layer - model, prompt, data, cache, or infra - by asking what changed and scoring each change on whether it explains the symptom and matches the onset timing. Tradeoff: the bisect gives you the most-likely layer fast, not a proof; you confirm by rolling that one change back. The 12.8 telemetry is what makes the bisect possible. Next: the levers that roll it back.

- A changelog (prompt deploy 20m, model bump 3d, reindex 1d) plus symptom toggles
- The bisect scores each change and narrows to the culprit layer

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Mitigate First: The Rollback Levers

### Mitigate First: The Rollback Levers

Restore users before you know the root cause. Five escalating levers - traffic (instant), model, cache, throttle, and the kill switch - each with its own blast radius and time-to-effect. Mitigate now; diagnose in the calm afterwards.

Here is the discipline that separates a calm responder from a panicked one: **mitigate first, diagnose second**. The moment you have a likely culprit, you roll it back to restore users — you do *not* keep the broken thing live while you hunt the exact bug. There are five escalating **rollback levers**. **Traffic**: shift 100 percent back to the previous revision — instant, zero-downtime, the default for a bad deploy (the 12.6 lever). **Model**: switch to a safe fallback model or provider when the current one is misbehaving. **Cache**: flush poisoned, stale, or PII-bearing entries and bump the cache-key version. **Throttle**: cap instances and concurrency to stop a cost runaway. And the **kill switch**: a feature flag that disables the AI path entirely and serves a degraded-but-safe fallback — the nuclear option for a systematic PII or safety leak. Graceful degradation beats a hard failure: a safe non-AI answer is better than a confidently wrong one. The block picks the right lever per incident, keyless.

> 🔌 **Analogy**
>
> It is the **circuit breakers on a fuse box**. When something in the house shorts out, you do not stand in the dark tracing wires with the power on — you flip the breaker for that circuit, the danger stops instantly, and *then* you investigate the faulty outlet in safety. A whole-house breaker (the kill switch) is there too, for when you cannot isolate the fault fast enough. Each rollback lever is a breaker at a different scope: one deploy, one model, the cache, the traffic, or the whole feature. Flip the smallest one that stops the danger.

A new deploy caused a quality regression and you have confirmed it. What is the fastest correct move?

**📝 `05_rollback_levers.py`** - *runnable*

In [ ]:
# Mitigate to restore users BEFORE you know the root cause. Five escalating levers.
LEVERS = {
    "traffic":  {"blast": "1 deploy",      "effect": "instant",   "use": "a bad deploy (quality/error regression)"},
    "model":    {"blast": "all requests",  "effect": "seconds",   "use": "the current model hallucinating"},
    "cache":    {"blast": "cached hits",   "effect": "seconds",   "use": "cache serving stale / wrong / PII"},
    "throttle": {"blast": "peak traffic",  "effect": "seconds",   "use": "a cost runaway / retry storm"},
    "kill":     {"blast": "the AI feature","effect": "instant",   "use": "a systematic PII / safety leak"},
}
def pick(incident):
    i = incident.lower()
    if "deploy" in i or "regression" in i:     return "traffic"
    if "hallucinat" in i:                       return "model"
    if "cache" in i or "stale" in i:            return "cache"
    if "cost" in i or "retry" in i or "budget" in i: return "throttle"
    if "pii" in i or "safety" in i:             return "kill"
    return "traffic"

print("The five rollback levers (mitigate first, diagnose second):")
for name, l in LEVERS.items():
    print("  {:<9} blast={:<13} effect={:<8} use when: {}".format(name, l["blast"], l["effect"], l["use"]))
print()
incidents = [
    "New deploy caused a quality regression",
    "Current model is hallucinating",
    "Semantic cache returning stale answers",
    "Cost runaway from a retry storm",
    "Systematic PII leak in outputs",
]
print("Pick the lever per incident:")
for inc in incidents:
    lv = pick(inc)
    print("  {:<42} -> {:<9} ({})".format(inc, lv, LEVERS[lv]["effect"]))
print()
print("Every lever restores users NOW; the root cause can wait for the calm after.")

# Output:
# The five rollback levers (mitigate first, diagnose second):
#   traffic   blast=1 deploy      effect=instant  use when: a bad deploy (quality/error regression)
#   model     blast=all requests  effect=seconds  use when: the current model hallucinating
#   cache     blast=cached hits   effect=seconds  use when: cache serving stale / wrong / PII
#   throttle  blast=peak traffic  effect=seconds  use when: a cost runaway / retry storm
#   kill      blast=the AI feature effect=instant  use when: a systematic PII / safety leak
#
# Pick the lever per incident:
#   New deploy caused a quality regression     -> traffic   (instant)
#   Current model is hallucinating             -> model     (seconds)
#   Semantic cache returning stale answers     -> cache     (seconds)
#   Cost runaway from a retry storm            -> throttle  (seconds)
#   Systematic PII leak in outputs             -> kill      (instant)
#
# Every lever restores users NOW; the root cause can wait for the calm after.

- Five **escalating levers**, each with a **blast radius** and a **time-to-effect**: traffic (one deploy, instant), model, cache, throttle, and the kill switch.
- Each incident maps to the **right lever**: a bad deploy → **traffic**; hallucination → **model**; stale cache → **cache**; cost runaway → **throttle**; PII leak → **kill**.
- Every lever **restores users now**; none of them require you to have found the exact root cause first.
- Graceful degradation is the theme: a safe fallback beats a confidently wrong answer — flip the smallest lever that stops the danger.

#### 💡 What just happened

⚡ What just happened? Mitigate before you diagnose: five escalating rollback levers - traffic (instant), model, cache, throttle, and the kill switch - each restore users without needing the root cause first. Tradeoff: a rollback may lose a good feature for a while, in exchange for stopping user harm now; you re-enable it after the calm diagnosis. Graceful degradation beats a hard failure. Next: the two runaways that need their own fast path.

- Five levers (traffic / model / cache / throttle / kill) with blast radius and time-to-effect
- Draw an incident card and pick the right lever; the wrong one over- or under-reacts

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The AI-Specific Runaways

### The AI-Specific Runaways

Two failure modes need a dedicated fast path. A retry storm amplifies - failures spawn retries that pile more load on a degraded provider and burn the budget - stopped by a circuit breaker and a budget cap. A PII leak’s blast radius includes the cache, so you scope and purge, then notify legal.

Two AI incidents are dangerous enough to memorize cold, because both *feed on themselves*. The first is the **retry storm**. When the provider degrades, your agents retry every failure — and each retry adds latency, burns credits, and piles *more* load onto an already-struggling provider, causing more failures, causing more retries. An agent stuck in a retry loop can drain the cloud budget in minutes, before any human notices. The fix is not more retries: it is a **circuit breaker** (stop hammering the failing service) plus a hard **budget or iteration cap**, with any hedge retries capped at a small fraction of base traffic. The second is the **PII or safety leak**, a SEV1 whose blast radius is bigger than it looks: because a response can be **cached and replayed**, one leaked answer can be served thousands of times, and you cannot know which other cached entries are also tainted. So you **scope** the affected serves, **purge the whole cache**, and notify legal or compliance — fixing the single response is not enough. The block simulates both, keyless.

> 🏦 **Analogy**
>
> The retry storm is a **bank run that feeds on itself**. A rumor that the bank is shaky sends everyone rushing to withdraw at once, which actually drains the bank, which confirms the rumor, which brings more people running. Blindly retrying a struggling provider is that run: the panic *causes* the collapse it feared. The circuit breaker is the bank closing its doors for an hour to break the panic. And the PII leak is a **printed newspaper**, not a spoken word: once the wrong answer is cached, it is on every doorstep, reprinted with every hit — you cannot un-say it one copy at a time; you have to recall the whole run.

Your provider is degraded and your agents retry every failure. Left alone, what happens to load and cost?

**📝 `06_runaways.py`** - *runnable*

In [ ]:
# TWO AI-specific runaways. (1) A retry storm amplifies: failures spawn retries, which
# add load, which cause more failures. A circuit breaker + budget cap flattens it.
BASE = 100          # steady request load
FAIL_RATE = 0.5     # half are failing (provider degraded)
RETRIES = 3         # each failure is retried 3x
COST_PER = 0.002    # $ per attempt

def run(breaker):
    load, total_cost = BASE, 0.0
    print("  tick  attempts  failures  retries_added  cost")
    for t in range(4):
        failures = int(load * FAIL_RATE)
        if breaker and failures > BASE * FAIL_RATE:   # breaker opens on a failure surge
            added = 0
            note = "  <- breaker OPEN: retries suppressed"
        else:
            added = failures * RETRIES
            note = ""
        total_cost += load * COST_PER
        print("  {:>3}   {:>7}   {:>7}   {:>12}   ${:>6.2f}{}".format(t, load, failures, added, load * COST_PER, note))
        load = BASE + added
    print("  total cost over 4 ticks: ${:.2f}".format(total_cost))

print("WITHOUT a circuit breaker (retries amplify the load):")
run(breaker=False)
print()
print("WITH a circuit breaker + budget cap (retries suppressed on a failure surge):")
run(breaker=True)
print()
# (2) A PII leak's blast radius includes the CACHE - a cached reply is replayed.
cache_entries = 5000
tainted_serves = 1240   # times the PII-bearing cached reply was served
print("PII leak - the blast radius is not one response:")
print("  the PII-bearing reply was cached and REPLAYED {} times".format(tainted_serves))
print("  and you cannot know which of the {} entries are also tainted".format(cache_entries))
print("  -> scope the {} affected serves AND flush the whole cache, then notify legal.".format(tainted_serves))

# Output:
# WITHOUT a circuit breaker (retries amplify the load):
#   tick  attempts  failures  retries_added  cost
#     0       100        50            150   $  0.20
#     1       250       125            375   $  0.50
#     2       475       237            711   $  0.95
#     3       811       405           1215   $  1.62
#   total cost over 4 ticks: $3.27
#
# WITH a circuit breaker + budget cap (retries suppressed on a failure surge):
#   tick  attempts  failures  retries_added  cost
#     0       100        50            150   $  0.20
#     1       250       125              0   $  0.50  <- breaker OPEN: retries suppressed
#     2       100        50            150   $  0.20
#     3       250       125              0   $  0.50  <- breaker OPEN: retries suppressed
#   total cost over 4 ticks: $1.40
#
# PII leak - the blast radius is not one response:
#   the PII-bearing reply was cached and REPLAYED 1240 times
#   and you cannot know which of the 5000 entries are also tainted
#   -> scope the 1240 affected serves AND flush the whole cache, then notify legal.

- **Without a circuit breaker**, the retry storm amplifies tick by tick — attempts climb from one hundred toward eight hundred, and the cost compounds.
- **With a breaker + budget cap**, the retries are suppressed on a failure surge, load stays bounded, and the total cost is roughly **half**.
- The **PII leak**’s blast radius is the cache: the leaked reply was replayed over a thousand times, and any entry could be tainted.
- So you **scope the affected serves and flush the whole cache**, then notify legal — fixing the one response would still leave the cache serving it.

#### 💡 What just happened

⚡ What just happened? Two runaways feed on themselves: a retry storm (failures spawn retries that amplify load and cost, stopped by a circuit breaker + budget cap) and a PII leak (whose blast radius includes the cache, so you scope and purge, not just fix one response). Tradeoff: the breaker sheds some legitimate retries to stop the storm, and a cache flush costs a cold spell, in exchange for stopping the runaway. Next: learning from all of it.

- A retry loop amplifies load and cost tick by tick, then a circuit breaker flattens it
- A PII leak whose blast radius fills every cached hit until a purge clears it

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Blameless Postmortem

### The Blameless Postmortem

Within a day of a SEV1/2, write a blameless postmortem: a timeline, the two clocks, the root-cause category with a version snapshot (model/prompt/index/guardrail), action items on the system, and a ‘where we got lucky’ near-miss. Blameless means you fix systems, not people.

The incident is over; now you make sure it does not happen again. Within a day or two of a SEV1 or SEV2, write a **blameless postmortem**. **Blameless** is the whole game: you focus on the systems, procedures, and training that let the incident happen — not on the person (or, in 2026, “the agent”) who shipped the change. The moment you assign blame, people stop being honest, and you lose the learning. A good AI postmortem has a **timeline**, the **impact**, the two clocks (**MTTD and MTTR**) reviewed to find the process bottleneck, and the root-cause **category** with a **version snapshot** — the model, prompt, retrieval-index, tool-schema, eval-dataset, and guardrail versions that were live — because “which version was running?” is almost always the crux of an AI incident. The **action items** get owners and due dates and fix the *system* (pin the model version, add a validator, alert on provider changes). And the most valuable section is **“where we got lucky”**: the near-miss that reveals what could have been far worse. The block builds one, keyless.

> ✈️ **Analogy**
>
> It is a **flight-crash investigation**. Aviation got extraordinarily safe not by firing pilots after every mistake, but by investigating *blamelessly*: the black box is read to reconstruct exactly what happened and change the *system* — the checklist, the training, the cockpit design — so the whole industry never repeats it. A crash board that spent its time deciding whose fault it was would learn nothing, because the next crew would hide their mistakes. The postmortem is your black box: reconstruct the timeline, snapshot the versions, fix the system, and treat the near-miss as seriously as the crash.

Your postmortem could either name the engineer who shipped the bad prompt, or record which prompt and model versions were live and what to change. Which makes you safer next time?

**📝 `07_postmortem.py`** - *runnable*

In [ ]:
# Within a day, write a BLAMELESS postmortem: timeline, the two clocks, the root-cause
# category with a VERSION SNAPSHOT, action items with owners, and "where we got lucky".
# Timeline in minutes from t0 (a provider model version bump went live at t0):
tl = {"began": 0, "detected": 240, "commander": 245, "root_cause": 260, "mitigated": 275, "resolved": 280}
mttd = tl["detected"] - tl["began"]
mttr = tl["resolved"] - tl["detected"]
duration = tl["resolved"] - tl["began"]

print("POSTMORTEM: hallucination spike after a model version bump")
print("  duration {}m   MTTD {}m   MTTR {}m".format(duration, mttd, mttr))
print()
print("  Timeline (minutes from t0):")
for k in ["began", "detected", "commander", "root_cause", "mitigated", "resolved"]:
    print("    t={:>3}  {}".format(tl[k], k.replace("_", " ")))
print()
print("  Root cause category: MODEL")
print("  Version snapshot (the crux of an AI postmortem):")
snapshot = {"model": "vendor-model v2 -> v3", "prompt": "p-1.4 (unchanged)", "index": "idx-11 (unchanged)",
            "guardrail": "g-2 (unchanged)"}
for k, v in snapshot.items():
    print("    {:<10} {}".format(k, v))
print()
print("  Action items (blameless - fix the system, not the person):")
for pri, item, owner in [("P0", "pin the exact model VERSION, not just the name", "platform"),
                          ("P0", "add numeric fact-checking to the output validator", "quality"),
                          ("P1", "alert on provider changelog / version changes", "sre")]:
    print("    {} {:<48} owner: {}".format(pri, item, owner))
print()
print("  Where we got lucky: the bad version only skewed numeric answers and was caught")
print("  before a larger launch - a near-miss that a slower MTTD would have made a headline.")

# Output:
# POSTMORTEM: hallucination spike after a model version bump
#   duration 280m   MTTD 240m   MTTR 40m
#
#   Timeline (minutes from t0):
#     t=  0  began
#     t=240  detected
#     t=245  commander
#     t=260  root cause
#     t=275  mitigated
#     t=280  resolved
#
#   Root cause category: MODEL
#   Version snapshot (the crux of an AI postmortem):
#     model      vendor-model v2 -> v3
#     prompt     p-1.4 (unchanged)
#     index      idx-11 (unchanged)
#     guardrail  g-2 (unchanged)
#
#   Action items (blameless - fix the system, not the person):
#     P0 pin the exact model VERSION, not just the name   owner: platform
#     P0 add numeric fact-checking to the output validator owner: quality
#     P1 alert on provider changelog / version changes    owner: sre
#
#   Where we got lucky: the bad version only skewed numeric answers and was caught
#   before a larger launch - a near-miss that a slower MTTD would have made a headline.

**📝 `oncall-runbook.py`** - *illustrative*

In [ ]:
# ON-CALL RUNBOOK - an illustrative minimal example (kill switch + auto-rollback).
import os, subprocess

# A feature flag you can flip WITHOUT a deploy - the fastest safe mitigation.
KILL_SWITCH = os.environ.get("AI_KILL_SWITCH", "off") == "on"

def answer(prompt: str) -> str:
    if KILL_SWITCH:
        # degrade gracefully: a safe non-AI fallback beats a confidently wrong AI answer
        return "Our assistant is briefly unavailable; here is our help center instead."
    return call_model(prompt)   # the normal AI path

# Auto-rollback: when the error rate stays over the SLO, shift traffic to the last good revision.
def auto_rollback(error_rate: float, threshold: float = 0.20) -> None:
    if error_rate <= threshold:
        return
    # the 12.6 traffic lever, run from the on-call machine:
    # gcloud run services update-traffic ai-svc --to-revisions LAST_GOOD=100
    subprocess.run(["gcloud", "run", "services", "update-traffic", "ai-svc",
                    "--to-revisions", "LAST_GOOD=100"], check=False)
    notify_oncall("auto-rollback fired: error rate over SLO, traffic shifted to LAST_GOOD")
# Output: (illustrative minimal example - needs gcloud + a deployed service and an on-call hook; not run here.)

- The **postmortem** reconstructs the timeline and reads off **MTTD, MTTR, and duration**, then names the root-cause **category** (MODEL, a version bump).
- The **version snapshot** — model, prompt, index, guardrail — is the crux: it shows exactly which version was live when it broke.
- **Action items** are blameless and fix the *system* (pin the model version, add a numeric validator, alert on provider changes), each with an owner.
- The **illustrative runbook** is the 3 AM artifact: a `KILL_SWITCH` feature flag that degrades gracefully, and an `auto_rollback` that shifts traffic to the last good revision when the error rate stays over the SLO.

#### 💡 What just happened

⚡ What just happened? A blameless postmortem records the timeline, the two clocks, the root-cause category with a version snapshot, action items that fix the system, and a ‘where we got lucky’ near-miss - blameless because blame kills the honesty you need. Tradeoff: writing it costs an hour or two after an exhausting incident, in exchange for never paying for the same incident twice. That is the whole lifecycle: detect, classify, command, triage, mitigate, and learn.

- Assemble the timeline (detect / mitigate / resolve) and read off MTTD / MTTR / duration
- Pick the root-cause category and reveal the version snapshot and the where-we-got-lucky near-miss

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Running an AI incident is a lifecycle, each stage with an AI-specific twist. It starts by rejecting the green-dashboard illusion: an AI incident is a **gray failure**, a clean 200 with a wrong answer, detected by the golden signals, not uptime (Step 1). You **classify** its severity and start the **MTTD/MTTR clock** — and for an AI app the long, silent MTTD is usually the real damage (Step 2). You **declare** the incident and assign the separated roles — an **Incident Commander** who coordinates, an **Ops lead** who executes, a **Comms lead** who owns the status page (Step 3). You **triage** by asking *what changed?*, bisecting to the model, prompt, data, cache, or infra with the 12.8 telemetry (Step 4). You **mitigate first** with the five rollback levers — traffic, model, cache, throttle, kill switch — restoring users before you know the root cause (Step 5). You keep a fast path for the two **runaways** — the retry storm (breaker + cap) and the PII leak (scope + purge) (Step 6). And you **learn** with a blameless postmortem: timeline, version snapshot, system fixes, and the near-miss (Step 7). Ask four questions of any incident response: was it **detected fast**, **commanded cleanly**, **mitigated before diagnosis**, and **learned from blamelessly**?

| Stage | The AI-specific twist | The tool |
|---|---|---|
| Detect | gray failures: 200-but-wrong | the golden signals (12.8) |
| Classify | quality drop = SEV2, the silent killer | severity + MTTD/MTTR |
| Command | one voice under chaos | IC / Ops / Comms roles |
| Triage | model / prompt / data / cache / infra | bisect by what changed |
| Mitigate | restore before root cause | the five rollback levers |
| Learn | which version was live? | blameless postmortem + snapshot |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now run an AI incident end to end. **Production readiness** — the operational maturity that makes incidents rarer and calmer — closes the module in Lesson 12.10, and cutting the **cost** that a runaway threatens is Module 13. The **eval set** a postmortem action item adds — turning a real incident into a permanent regression test — is Lesson 14.4, and the **observability** that detects the incident and drives the triage is Lesson 12.8. The primary references are the [Google SRE incident management guide](https://sre.google/resources/practices-and-processes/incident-management-guide/), the [SRE postmortem-culture chapter](https://sre.google/sre-book/postmortem-culture/), [Rootly on the Incident Commander role](https://rootly.com/incident-response/incident-commander), [the PagerDuty incident-response docs](https://response.pagerduty.com/), and the [SRE workbook](https://github.com/google/site-reliability-workbook).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **ProductionIncidents**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_9.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.9.html` - regenerate this notebook whenever the source HTML is updated.*
